In [1]:
import requests
import time

# URL de l'API Infoclimat utilisée pour récupérer les données météorologiques.
# Cette API permet d'extraire des observations historiques par station.
url = "https://www.infoclimat.fr/opendata/"

# Liste des stations sélectionnées après la phase d'analyse et de filtrage.
# Ces stations correspondent aux capteurs exploitables situés à Paris.
liste_stations = ["000BV", "000CT", "000EW", "ME098"]

# Année choisie pour la constitution du jeu de données.
annee_extraction = "2025"

# Définition des saisons pour l'année 2025 (Format AAAA-MM-JJ).
# Pour chaque station génère moi un fichier CSV contenant les données relevées d'une saison.
# Le découpage par saison permet :
    # De contourner les limitations de l'API.
    # De garantir une représentation équilibrée des conditions météorologiques.
saisons = {
    "hiver": {
        "debut": f"{annee_extraction}-01-01",
        "fin":   f"{annee_extraction}-03-31"
    },
    "printemps": {
        "debut": f"{annee_extraction}-04-01",
        "fin":   f"{annee_extraction}-06-30"
    },
    "ete": {
        "debut": f"{annee_extraction}-07-01",
        "fin":   f"{annee_extraction}-09-30"
    },
    "automne": {
        "debut": f"{annee_extraction}-10-01",
        "fin":   f"{annee_extraction}-12-31"
    }
}

# Parcours de chaque saison afin de segmenter l'extraction.
# Permet de répartir les appels API et d'éviter les limitations du volume :
for saison, dates in saisons.items():
    debut_saison = dates["debut"]
    fin_saison = dates["fin"]

    print(f"\n--- Traitement de la saison : {saison} ({debut_saison} au {fin_saison}) ---")

    # Pourquoi une boucle ici : Besoin de voir la structure du fichier par station dans un premier temps + besoin de limiter la quantité de données récoltées.
    for station in liste_stations:
        # Paramètres de la requête API :
        params = {
            "method": "get",
            "format": "csv",
            "stations[]": station,
            "start": debut_saison,
            "end": fin_saison,
            "token": "xxxxx",
        }

        try:
            # Envoi de la requête à l'API :
            response = requests.get(url, params=params)

            # Vérification du succès de la requête :
            if response.status_code == 200:

                # Récupération du contenu CSV:
                csv_content = response.text

                # Nom du fichier : donnees_meteo_<station>_<saison>.csv:
                filename = f"donnees_meteo_{station}_{saison}_{annee_extraction}.csv"

                # Sauvegarde du fichier localement.
                # Le format CSV est utilisé pour faciliter l'inspection et les traitements ultérieurs :
                with open(filename, mode='w', encoding='utf-8') as file:
                    file.write(csv_content)

                print(f"Succès : {filename} généré ({len(csv_content)} octets).")

            else:
                # Gestion des erreurs côté API :
                print(f"Erreur API pour {station} ({saison}) : {response.status_code} - {response.text}")

        except Exception as e:
            # Gestion des erreurs réseau ou exceptions :
            print(f"Erreur de connexion pour {station} : {e}")

        # PAUSE DE 5sec Sauf si c'est la dernière station pour gagner du temps à la fin.
        # Pause necessaire car la quantité de données récupérable est limitée en durée d'exécution (env 3sec) :
        if station != liste_stations[-1]:
            time.sleep(5)

    # Pause supplémentaire entre les saisons pour réduire la charge globale envoyée à l'API :
    if saison != list(saisons.keys())[-1]:
        print(f"Pause de 10 secondes avant la saison suivante.")
        time.sleep(10)

# Message de fin du processus :
print("\nTous les téléchargements sont terminés.")


--- Traitement de la saison : hiver (2025-01-01 au 2025-03-31) ---
Succès : donnees_meteo_000BV_hiver_2025.csv généré (685128 octets).
Succès : donnees_meteo_000CT_hiver_2025.csv généré (237202 octets).
Succès : donnees_meteo_000EW_hiver_2025.csv généré (653867 octets).
Succès : donnees_meteo_ME098_hiver_2025.csv généré (661118 octets).
Pause de 10 secondes avant la saison suivante.

--- Traitement de la saison : printemps (2025-04-01 au 2025-06-30) ---
Succès : donnees_meteo_000BV_printemps_2025.csv généré (718658 octets).
Succès : donnees_meteo_000CT_printemps_2025.csv généré (245782 octets).
Succès : donnees_meteo_000EW_printemps_2025.csv généré (676117 octets).
Succès : donnees_meteo_ME098_printemps_2025.csv généré (707519 octets).
Pause de 10 secondes avant la saison suivante.

--- Traitement de la saison : ete (2025-07-01 au 2025-09-30) ---
Succès : donnees_meteo_000BV_ete_2025.csv généré (733388 octets).
Succès : donnees_meteo_000CT_ete_2025.csv généré (251062 octets).
Succès :